In [2]:
import torch
torch.cuda.is_available()

True

In [3]:
!pip install datasets transformers scikit-learn pandas numpy

In [6]:
from datasets import load_dataset

toxic_dataset = load_dataset("civil_comments")

toxic_dataset

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/194M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/187M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/20.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1804874 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/97320 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/97320 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'toxicity', 'severe_toxicity', 'obscene', 'threat', 'insult', 'identity_attack', 'sexual_explicit'],
        num_rows: 1804874
    })
    validation: Dataset({
        features: ['text', 'toxicity', 'severe_toxicity', 'obscene', 'threat', 'insult', 'identity_attack', 'sexual_explicit'],
        num_rows: 97320
    })
    test: Dataset({
        features: ['text', 'toxicity', 'severe_toxicity', 'obscene', 'threat', 'insult', 'identity_attack', 'sexual_explicit'],
        num_rows: 97320
    })
})

In [7]:
import pandas as pd

# Convert training split to DataFrame
toxic_df = pd.DataFrame(toxic_dataset["train"])

# Combine toxicity-related columns into single toxicity label
toxicity_columns = [
    "toxicity",
    "severe_toxicity",
    "obscene",
    "threat",
    "insult",
    "identity_attack",
    "sexual_explicit"
]

# If any of these columns > 0.5 → toxic
toxic_df["toxicity_label"] = (toxic_df[toxicity_columns].max(axis=1) > 0.5).astype(int)

# Keep only text + toxicity_label
toxic_df = toxic_df[["text", "toxicity_label"]]

# Sample 6000 rows for manageable training
toxic_sample = toxic_df.sample(6000, random_state=42)

toxic_sample.head()

,text,toxicity_label
286892,What a breathe of fresh air to have someone wh...,0
419218,Your jewish friends were the ones who told you...,1
1055330,Possible collusion by Trump and his affiliates...,0
1382764,Exactly. We need a % of GDP spending cap at t...,0
256049,"By your own comment, even if some of them vote...",0


In [8]:
from datasets import load_dataset

depression_dataset = load_dataset("dreaddit")

depression_dataset

DatasetNotFoundError: Dataset 'dreaddit' doesn't exist on the Hub or cannot be accessed.

In [9]:
from datasets import load_dataset

emotion_dataset = load_dataset("go_emotions")

emotion_dataset

README.md: 0.00B [00:00, ?B/s]

simplified/train-00000-of-00001.parquet:   0%|          | 0.00/2.77M [00:00<?, ?B/s]

simplified/validation-00000-of-00001.par(…):   0%|          | 0.00/350k [00:00<?, ?B/s]

simplified/test-00000-of-00001.parquet:   0%|          | 0.00/347k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/43410 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5426 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5427
    })
})

In [10]:
emotion_dataset["train"].features["labels"]

List(ClassLabel(names=['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']))

In [11]:
import pandas as pd

# Convert GoEmotions train split to DataFrame
emotion_df = pd.DataFrame(emotion_dataset["train"])

# Define depression-related emotion indices
depression_indices = [16, 25, 9, 24]  # grief, sadness, disappointment, remorse

# Create depression label
def detect_depression(label_list):
    return int(any(label in depression_indices for label in label_list))

emotion_df["depression_label"] = emotion_df["labels"].apply(detect_depression)

# Keep only text + depression_label
emotion_df = emotion_df[["text", "depression_label"]]

# Sample 6000 rows
depression_sample = emotion_df.sample(6000, random_state=42)

depression_sample.head()

,text,depression_label
25759,The only way this works is if [NAME] is doing ...,0
22531,Access should be hindered it's getting destroyed.,0
18418,Totally fair. All I was trying to remind every...,0
31117,"I'm poly and jn the Raleigh area too, moved he...",0
5733,Naw man Asain men have an easier time. Most of...,0


In [12]:
# --- Prepare Toxic Dataset ---

toxic_sample = toxic_sample.copy()
toxic_sample.rename(columns={"toxicity_label": "toxicity"}, inplace=True)

# Add missing columns
toxic_sample["depression"] = 0
toxic_sample["manipulation"] = 0

toxic_sample = toxic_sample[["text", "depression", "toxicity", "manipulation"]]


# --- Prepare Depression Dataset ---

depression_sample = depression_sample.copy()
depression_sample.rename(columns={"depression_label": "depression"}, inplace=True)

# Add missing columns
depression_sample["toxicity"] = 0
depression_sample["manipulation"] = 0

depression_sample = depression_sample[["text", "depression", "toxicity", "manipulation"]]


# --- Combine Both ---

combined_df = pd.concat([toxic_sample, depression_sample], ignore_index=True)

combined_df.head()

,text,depression,toxicity,manipulation
0,What a breathe of fresh air to have someone wh...,0,0,0
1,Your jewish friends were the ones who told you...,0,1,0
2,Possible collusion by Trump and his affiliates...,0,0,0
3,Exactly. We need a % of GDP spending cap at t...,0,0,0
4,"By your own comment, even if some of them vote...",0,0,0


In [13]:
# Define manipulation-related phrases
manipulation_keywords = [
    "if you loved me",
    "you owe me",
    "it's your fault",
    "after everything i did",
    "you made me do this",
    "don't leave me",
    "no one else will love you",
    "you need me",
    "i sacrificed for you",
    "you can't survive without me"
]

# Function to detect manipulation
def detect_manipulation(text):
    text = text.lower()
    return int(any(keyword in text for keyword in manipulation_keywords))

# Apply to dataset
combined_df["manipulation"] = combined_df["text"].apply(detect_manipulation)

combined_df.head()

,text,depression,toxicity,manipulation
0,What a breathe of fresh air to have someone wh...,0,0,0
1,Your jewish friends were the ones who told you...,0,1,0
2,Possible collusion by Trump and his affiliates...,0,0,0
3,Exactly. We need a % of GDP spending cap at t...,0,0,0
4,"By your own comment, even if some of them vote...",0,0,0


In [14]:
combined_df[["depression", "toxicity", "manipulation"]].sum()

,0
depression,407
toxicity,336
manipulation,1


In [15]:
combined_df[["depression", "toxicity", "manipulation"]].mean()

,0
depression,0.033917
toxicity,0.028000
manipulation,0.000083


In [16]:
# Create synthetic manipulation sentences

manipulation_texts = [
    "If you really loved me, you would do this for me.",
    "After everything I did for you, this is how you treat me?",
    "You owe me for all the sacrifices I made.",
    "If you leave me, I will ruin myself and it will be your fault.",
    "No one else will ever love you the way I do.",
    "You made me behave this way.",
    "If you cared, you would agree with me.",
    "I sacrificed my life for you, you can't leave now.",
    "You need me more than I need you.",
    "Without me, you are nothing."
]

# Duplicate them multiple times to increase count
synthetic_manipulation = []

for text in manipulation_texts:
    for _ in range(30):  # 30 duplicates each
        synthetic_manipulation.append({
            "text": text,
            "depression": 0,
            "toxicity": 0,
            "manipulation": 1
        })

synthetic_df = pd.DataFrame(synthetic_manipulation)

# Add to combined dataset
combined_df = pd.concat([combined_df, synthetic_df], ignore_index=True)

combined_df[["depression", "toxicity", "manipulation"]].sum()

,0
depression,407
toxicity,336
manipulation,301


In [17]:
combined_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)

combined_df.head()

,text,depression,toxicity,manipulation
0,Especially the democrats. What do you think w...,0,0,0
1,You get a creepy dog biscuit,0,0,0
2,they sounds like autistic meltdowns triggered ...,0,0,0
3,Unfortunately that's what happens most often w...,0,0,0
4,I'd like to apologize on behalf of all Azwel m...,0,0,0


In [18]:
from sklearn.model_selection import train_test_split

# First split: Train (80%) and Temp (20%)
train_df, temp_df = train_test_split(
    combined_df,
    test_size=0.2,
    random_state=42
)

# Second split: Validation (10%) and Test (10%)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42
)

print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))

Train size: 9840
Validation size: 1230
Test size: 1230


In [19]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [20]:
# Create folder if not exists
import os

save_path = "/content/drive/MyDrive/DeepSignal"
os.makedirs(save_path, exist_ok=True)

# Save splits
train_df.to_csv(f"{save_path}/train.csv", index=False)
val_df.to_csv(f"{save_path}/val.csv", index=False)
test_df.to_csv(f"{save_path}/test.csv", index=False)

print("Datasets saved successfully.")

Datasets saved successfully.
